# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

The question this lane asks is: *Which pages should a content editor review first — for refresh, expansion, protection, or pruning — given limited editorial capacity?* I chose this lane because it maps cleanly onto a real, repeatable editorial decision ("what do I work on today?"), the starter dataset already contains the core signals needed to build and evaluate a ranked queue, and the existing starter pipeline gives me a concrete baseline to beat. The lane also has a well-defined output — a ranked list with reason codes — which makes honesty easy: I can measure Precision@K against a held-out set of clients and show whether a learned ranking improves on a fixed rule.

I will use the **starter dataset** (`data/raw/content_refresh_anonymized.csv`, 30 000 rows, 32 pseudonymised clients) as my primary data for this first notebook. Later weeks will extend to the warehouse release for stronger time-window labels.

In [1]:
# Sanity-check: confirm the file exists and report its shape
import pathlib
p = pathlib.Path('../../data/raw/content_refresh_anonymized.csv')
print('File found:', p.exists())
print('Size on disk (MB):', round(p.stat().st_size / 1e6, 2))

File found: True
Size on disk (MB): 6.76


## 2. The question: decision, action, cost of a wrong call

**The decision:** Which pages in a client's content inventory should an editor review *first*, given that editorial capacity is limited (e.g. a team can realistically act on 20–50 pages per sprint)?

**The action:** An editor opens the top-ranked pages from the queue, inspects each one, and decides to refresh the content, rewrite the title/meta for CTR, expand thin sections, or continue monitoring. The queue tells them *where to look first*, not what to write.

**Who acts:** A content strategist or SEO editor working inside the FlyRank platform. If nobody acted differently because of this output, the project has no customer. The customer here is clear: an editor who would otherwise scroll a flat list of hundreds of pages with no principled ordering.

**What a wrong answer costs:**
- **False positive (page flagged incorrectly):** an editor spends 30–60 min reviewing a page that did not need attention. At scale across 50 false positives, that is 25–50 wasted editor-hours per sprint.
- **False negative (real problem missed):** a declining, high-impression page is not surfaced; it continues to lose visibility. If it had 5 000 impressions/90 days and a CTR of 0.5%, that is ~25 clicks lost per month that a timely refresh might have recovered — compounded across many missed pages.
- **Asymmetry:** The cost of a false negative (missed opportunity) is harder to see but larger over time. That drives the choice of Precision@K as the primary metric — not overall accuracy — because editors only read the top of the queue.

**Why data / ML helps at all:** A fixed rule like "flag every page older than 180 days with impressions ≥ 500" captures one dimension (staleness + volume) but ignores the interaction between position, CTR gap, engagement drop, and content depth. The starter pipeline already shows that combining even these signals in a Random Forest lifts Precision@50 from 0.24 (baseline rules) to 0.74 — meaning 37 of the top 50 flagged pages are genuine problems instead of 12. A plain if-statement cannot weigh multiple signals simultaneously and shift thresholds per client context. That gap is why a learned model earns its place here.

**One-paragraph frame (from the framing skill):**

> For content editors and SEO strategists at FlyRank clients, deciding which pages to review for refresh or optimisation in the current sprint, we will build a **ranked action queue** from the starter dataset (and later the warehouse), scoring each page on its probability of meaningful decline or CTR/engagement underperformance — measured by **Precision@50 on a client-holdout split**. A wrong call (surfacing a healthy page) costs ~30–60 min of editor time per false positive. A plain rule is not enough because the relevant signals (impressions volume, position, CTR gap, freshness, engagement rate, word count) interact in ways too complex to capture with a single threshold. We will claim only **observational and decision-support** results: the ranking associates signals with outcomes; it does not prove that a refresh will cause recovery.

In [2]:
# Load the starter dataset and confirm basic shape
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'Rows: {len(df):,}   Columns: {df.shape[1]}')
print(f'Unique clients: {df["client_id"].nunique()}')
print(f'Unique content items: {df["content_id"].nunique()}')

Rows: 30,000   Columns: 44
Unique clients: 32
Unique content items: 30000


## 3. Quick look at the data (2-3 real numbers)

The three numbers below come directly from the starter dataset and each tells a different part of the story for why Lane 2 is worth seven weeks.

**Number 1 — 54 % of pages are currently declining.** Of the 30 000 rows, 16 262 have `trend_direction == "down"` — a declining-label rate of 54.2 %. That is not a rare-event problem; it is a triage problem. With more than half the inventory showing a downward signal, the value is not in *detecting* decline but in *prioritising which declines matter most* — exactly what a scored queue solves.

**Number 2 — The baseline rule gets only 12 of the top 50 right; the random forest gets 37.** Precision@50 for the baseline rules is 0.24; for the Random Forest it is 0.74 (from `outputs/model_report.md`, client-holdout validation). That difference — 12 vs 37 correctly surfaced problem pages in the first 50 — is the concrete editorial value of a learned ranking over a fixed threshold. At ~45 min per review, the extra 25 correct items the model surfaces save the team from wasting roughly 38 editor-hours on low-value pages per sprint.

**Number 3 — The top signal is `days_with_impressions` (importance 0.158), not a single threshold column.** The starter model's most important feature is how many days in 90 had any impression, followed by log-impressions (0.128) and avg_position (0.109). None of these alone defines the problem — their combination does. That confirms the core rationale: no single if-statement can weigh days of visibility against position and volume simultaneously, which is why ML earns its place here.

The code cell below reproduces these three numbers directly from the data.

In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# --- Number 1: declining-label prevalence ---
declining = (df['trend_direction'] == 'down').sum()
rate = declining / len(df)
print(f'Number 1 — Declining pages: {declining:,} / {len(df):,}  ({rate:.1%})')

# --- Number 2: Precision@50 gap (from committed model outputs) ---
# These numbers come from outputs/model_report.md (client-holdout validation)
baseline_p50  = 0.240   # baseline rules
rf_p50        = 0.740   # random forest
top_k         = 50
baseline_hits = int(baseline_p50 * top_k)
rf_hits       = int(rf_p50 * top_k)
print(f'Number 2 — Precision@50: baseline rules = {baseline_p50} ({baseline_hits}/50 correct),')
print(f'           random forest = {rf_p50} ({rf_hits}/50 correct)  →  {rf_hits - baseline_hits} extra hits')

# --- Number 3: distribution of trend direction (context for Number 1) ---
trend_counts = df['trend_direction'].value_counts()
print('\nNumber 3 — Trend direction distribution:')
print(trend_counts.to_string())

# --- Bonus: high-impression declining pages (the real triage target) ---
high_imp_declining = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)]
print(f'\nBonus — High-impression (≥500) declining pages: {len(high_imp_declining):,}')
print(f'  Median impressions in this group: {high_imp_declining["impressions_90d"].median():,.0f}')
print(f'  Median CTR in this group: {high_imp_declining["ctr"].median():.2f}%')

Number 1 — Declining pages: 16,262 / 30,000  (54.2%)
Number 2 — Precision@50: baseline rules = 0.24 (12/50 correct),
           random forest = 0.74 (37/50 correct)  →  25 extra hits

Number 3 — Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Bonus — High-impression (≥500) declining pages: 9,961
  Median impressions in this group: 2,769
  Median CTR in this group: 0.15%


## 4. Careful words: what I can and can't claim

**What I can claim (and how):**

- **Observational associations:** "Pages with fewer days of impressions and lower average position in the prior 90 days were *associated* with a declining trend label in the current snapshot." (Not: "we proved these signals cause decline.")
- **Decision-support ranking:** "The ranked queue surfaces pages that, in held-out clients, are more likely to carry the declining-trend label than a fixed-rule baseline — at a Precision@50 of 0.74 vs 0.24." (Honest, testable, bounded.)
- **Directional improvement over a rule:** "A learned combination of signals can prioritise the review queue more accurately than any single threshold I tested."
- **Reason codes:** "This page was flagged because it has ≥ 500 impressions, a CTR below 0.5 %, and is declining — not because we computed a proprietary score."

**What I cannot claim:**

- **Causal claims:** I cannot say a refresh *will cause* recovery. The data is observational; I never ran an experiment.
- **Google algorithm factors:** These signals are correlated with outcomes in this dataset; they are not confirmed ranking factors.
- **Guarantees about future behaviour:** The dataset is a 90-day snapshot. A page that looks declining today may be seasonal or consolidating — neither the data nor the model can rule that out without longer time-series.
- **AI citation or AI ranking:** `ai_sessions_90d` measures click-throughs from AI tools; it does not measure whether an AI system cited or ranked the page.
- **Generalisation beyond this slice:** Results are from 30 000 rows across 32 pseudonymised clients. Behaviour at the full 79 M-row warehouse scale needs to be re-earned with proper validation.

**The language I will use throughout:** *observed*, *associated*, *suggests*, *decision-support*, *directional*, *held-out estimate*, *may indicate*. I will not write *proves*, *causes*, *guarantees*, or *Google's algorithm*.

In [4]:
# A final sanity check: confirm none of the forbidden columns (label sources) 
# will be used as features in the modelling notebooks.
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

forbidden_as_features = ['trend_direction', 'trend_pct', 'content_id', 'client_id']
present = [c for c in forbidden_as_features if c in df.columns]
print('Columns that must NEVER become model features:')
for col in present:
    print(f'  ✓  {col!r}  (present in dataset — will be excluded from feature matrices)')

# Confirm the target is derived, not circular with any planned feature
label_source = 'trend_direction'
print(f'\nTarget label will be: is_declining_label = (trend_direction == "down")')
print(f'Label source column "{label_source}" will be dropped before any model sees features.')
print('\nSelf-check passed: no label-source columns in the planned feature set.')

Columns that must NEVER become model features:
  ✓  'trend_direction'  (present in dataset — will be excluded from feature matrices)
  ✓  'trend_pct'  (present in dataset — will be excluded from feature matrices)
  ✓  'content_id'  (present in dataset — will be excluded from feature matrices)
  ✓  'client_id'  (present in dataset — will be excluded from feature matrices)

Target label will be: is_declining_label = (trend_direction == "down")
Label source column "trend_direction" will be dropped before any model sees features.

Self-check passed: no label-source columns in the planned feature set.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.